In [13]:
import sys


import pandas, numpy, scipy, sklearn


In [14]:
import re
from pathlib import Path


BASE = Path.cwd().parent
RAW = BASE / "Datasets" / "raw"



In [15]:
def parse_wos_file(path):
    """
    Read one Web of Science export file and return a list of records.

    WoS exports are field-tagged, not CSV. A two-letter tag starts each
    field. Continuation lines begin with three spaces. "ER" ends a record,
    "EF" ends the file.

        TI A paper about something
           that wraps onto a second line
        WC Computer Science, Artificial Intelligence; Engineering, Electrical
        UT WOS:000123456700001
        ER

    Each record becomes a dictionary mapping tag -> list of lines.
    """
    records = []
    current = {}
    current_tag = None

    # utf-8-sig strips the byte-order mark WoS puts at the file start
    with open(path, encoding="utf-8-sig") as f:
        for line in f:
            line = line.rstrip("\n")

            if line.startswith("ER"):            # end of one record
                if current:
                    records.append(current)
                current, current_tag = {}, None
                continue

            if line.startswith("EF"):            # end of file
                break

            match = re.match(r"^([A-Z][A-Z0-9]) (.*)$", line)
            if match:                            # a new field starts
                current_tag = match.group(1)
                current.setdefault(current_tag, []).append(match.group(2))
            elif line.startswith("   ") and current_tag:
                current[current_tag].append(line[3:])   # continuation line

    return records

In [16]:
files = sorted(RAW.glob("*.txt"))
print(f"found {len(files)} files")

test = parse_wos_file(files[0])
print(f"{files[0].name}: {len(test)} records")

found 20 files
Biotechnology & Applied Microbiology-1.txt: 500 records


In [17]:
record = test[0]

print("tags present:", sorted(record.keys()))
print()
print("title   :", " ".join(record.get("TI", []))[:90])
print("journal :", " ".join(record.get("SO", [])))
print("year    :", " ".join(record.get("PY", [])))
print("WC      :", " ".join(record.get("WC", [])))
print("UT      :", " ".join(record.get("UT", [])))
print("refs    :", len(record.get("CR", [])), "cited references")
print()
print("first 3 references:")
for r in record.get("CR", [])[:3]:
    print("   ", r)

tags present: ['AB', 'AF', 'AU', 'BP', 'C1', 'C3', 'CR', 'DA', 'DI', 'DT', 'EI', 'EM', 'EP', 'FN', 'FU', 'FX', 'GA', 'ID', 'IS', 'J9', 'JI', 'LA', 'NR', 'OA', 'OI', 'PA', 'PD', 'PG', 'PI', 'PM', 'PT', 'PU', 'PY', 'RI', 'RP', 'SC', 'SN', 'SO', 'TC', 'TI', 'U1', 'U2', 'UT', 'VL', 'VR', 'WC', 'WE', 'Z9']

title   : BioBERT: a pre-trained biomedical language representation model for biomedical text mining
journal : BIOINFORMATICS
year    : 2020
WC      : Biochemical Research Methods; Biotechnology & Applied Microbiology; Computer Science, Interdisciplinary Applications; Mathematical & Computational Biology; Statistics & Probability
UT      : WOS:000518528800035
refs    : 38 cited references

first 3 references:
    Alsentzer Emily, 2019, [No title captured], P72, DOI [10.18653/v1/w19-1909, DOI 10.18653/V1/W19-1909, 10.18653/v1/W19-1909]
    Ananiadou S, 2013, P LBM 2013, P39
    [Anonymous], J CHEMINFORM


In [18]:
total = 0
for path in files:
    records = parse_wos_file(path)
    total += len(records)
    print(f"{path.name:<45} {len(records):>5}")

print(f"\n{len(files)} files, {total} records total")
print("Expect ~500 per file (WoS export cap), 1000 per field, 10000 overall.")

Biotechnology & Applied Microbiology-1.txt      500
Biotechnology & Applied Microbiology-2.txt      500
CS AI-1.txt                                     500
CS AI-2.txt                                     500
CS Information Systems-1.txt                    500
CS Information Systems-2.txt                    500
Chemistry, Multidisciplinary-1.txt              500
Chemistry, Multidisciplinary-2.txt              500
Engineering, Electrical & Electronic-1.txt      500
Engineering, Electrical & Electronic-2.txt      500
Environmental Sciences-1.txt                    500
Environmental Sciences-2.txt                    500
Materials Science, Multidisciplinary-1.txt      500
Materials Science, Multidisciplinary-2.txt      500
Mathematics-1.txt                               500
Mathematics-2.txt                               500
Physics, Applied-1.txt                          500
Physics, Applied-2.txt                          500
Psychology, Multidisciplinary-1.txt             500
Psychology, 